# Notebook to figure out how to define the pipeline to generate singlecell embeddings
We will use scGPT to encode our single cell data initially
## Imports

In [1]:
%load_ext autoreload
%autoreload 2
import scanpy as sc
import sys
import numpy as np
import pandas as pd
import anndata as ad

# Load packages and classes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tiffslide
import seaborn as sns
import gget
import tifffile
import zarr

# MosaicDataset and BruceDataset classes allow loading and visualisation of the different data sources
from gbmhackathon import MosaicDataset
from gbmhackathon.data.io.loaders import SingleCellLoader
from gbmhackathon.s3_loader import get_s3_dataset_info

## Clone scFoundation repo

In [2]:
%cd ..

/home/sagemaker-user/gbm_hackathon


In [3]:
!git clone https://github.com/biomap-research/scFoundation.git

Cloning into 'scFoundation'...
remote: Enumerating objects: 484, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 484 (delta 22), reused 14 (delta 6), pack-reused 444 (from 1)
Receiving objects: 100% (484/484), 107.04 MiB | 36.88 MiB/s, done.
Resolving deltas: 100% (291/291), done.
Updating files: 100% (364/364), done.


Git clone scFoundation in scripts. Once it is cloned, modifiy line 59 and 116 of scFoundation/model/get_embedding.py as follows:

- line 59: `gene_list_df = pd.read_csv('../scFoundation/OS_scRNA_gene_index.19264.tsv', header=0, delimiter='\t')`
- line 116: `ckpt_path = '../scFoundation/model/models/models.ckpt'`

## Retrieve SingleCell data

In [ ]:
# Note that it can take up to 12 minutes to load the single-cell data because it is heavy
adata = MosaicDataset.load_singlecell("pa-3dqtp2dd4t56b7jvg-bx3h881cqf4ushomn6mknw3cstocaeuw1b-s3alias")
# Display the content of the anndata object
adata.__dict__.keys()

In [ ]:
sc.pl.umap(adata)

## Ensure correct genes names

In [ ]:
import mygene
import pandas as pd
import anndata as ad

# Initialize the MyGeneInfo object
mg = mygene.MyGeneInfo()

# Assuming adata is your AnnData object and is already loaded
# Example: adata = ad.read_h5ad('path_to_your_anndata.h5ad')

# Convert Ensembl IDs in adata to a list
ensembl_ids = adata.var_names.to_list()

# Query MyGeneInfo for the gene symbols
gene_info = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human', as_dataframe=True)

# Handle duplicate hits by keeping the first occurrence
gene_info = gene_info[~gene_info.index.duplicated(keep='first')]

# Print the columns and the first few rows of the DataFrame for debugging
print("Columns in gene_info DataFrame:", gene_info.columns)
print(gene_info.head())

# Ensure there are no missing Ensembl IDs in adata.var_names
missing_ids = set(ensembl_ids) - set(gene_info.index)
if missing_ids:
    print(f"Warning: {len(missing_ids)} Ensembl IDs not found in gene_info")
    
# Map the Ensembl IDs in the AnnData object to gene symbols
adata.var['symbol'] = adata.var_names.map(gene_info['symbol'])

# Update the AnnData object's index to use gene symbols
adata.var_names = adata.var['symbol']

In [ ]:
# Remove duplicate gene names
def remove_duplicate_var_indices(adata):
    # Extract the var index
    var_index = adata.var.index
    
    # Find duplicates
    duplicates = var_index[var_index.duplicated(keep='first')]
    
    # Drop duplicates
    adata = adata[:, ~var_index.isin(duplicates)]
    
    # Optionally print the number of removed variables
    print(f"Removed {len(duplicates)} variables with duplicate indices.")
    
    return adata

# Apply the function to your AnnData object
adata = remove_duplicate_var_indices(adata)

In [ ]:
# Ensure no variable indices are NaNs
def replace_nan_var_indices(adata):
    # Replace NaN in the var index with a placeholder
    adata.var.index = adata.var.index.fillna('unknown')
    
    # Ensure indices are unique after replacement
    if not adata.var.index.is_unique:
        raise ValueError("Indices are still not unique after replacing NaN.")
    
    return adata

# Apply the function to your AnnData object
adata = replace_nan_var_indices(adata)

## Convert AnnData to DataFrame

In [ ]:
def prepare_adata_for_selection(adata):
    # Convert the AnnData matrix to a DataFrame
    X_df = pd.DataFrame(
        adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
        index=adata.obs_names,
        columns=adata.var_names
    )
    return X_df

In [ ]:
X_df = prepare_adata_for_selection(adata)

## Reindexing gene names

In [ ]:
def main_gene_selection(X_df, gene_list):
    """
    Describe:
        rebuild the input adata to select target genes encode protein 
    Parameters:
        adata->`~anndata.AnnData` object: adata with var index_name by gene symbol
        gene_list->list: wanted target gene 
    Returns:
        adata_new->`~anndata.AnnData` object
        to_fill_columns->list: zero padding gene
    """
    to_fill_columns = list(set(gene_list) - set(X_df.columns))
    padding_df = pd.DataFrame(np.zeros((X_df.shape[0], len(to_fill_columns))), 
                              columns=to_fill_columns, 
                              index=X_df.index)
    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns, var

In [ ]:
gene_list_df = pd.read_csv('OS_scRNA_gene_index.19264.tsv', header=0, delimiter='\t')
gene_list = list(gene_list_df['gene_name'])
X_df, to_fill_columns, var = main_gene_selection(X_df, gene_list)

## Getting embeddings

In [ ]:
# Load the prepared csv file
X_df = sc.read_csv('undersampled.csv')
X_df = X_df.to_df()

In [ ]:
!python scfoundation/model/get_embedding.py --task_name Baron --input_type singlecell --output_type cell --pool_type all --tgthighres a5 --data_path undersampled.csv --save_path ./ --pre_normalized F --version rde

In [ ]:
# Rename the output .npy file to undersampled.npy
refdf = X_df
imputeemb = np.load(f'embedding.npy')
imputeAdata = sc.AnnData(pd.DataFrame(imputeemb, index=refdf.index))
sc.pp.scale(imputeAdata)
sc.tl.pca(imputeAdata)
sc.pp.neighbors(imputeAdata)
sc.tl.umap(imputeAdata)

In [ ]:
def transfer_obs(original_adata, subsampled_adata):
    common_indices = subsampled_adata.obs.index.intersection(original_adata.obs.index)
    subsampled_adata.obs = original_adata.obs.loc[common_indices].copy()
    return subsampled_adata

In [ ]:
imputeAdata = transfer_obs(adata, imputeAdata)

In [ ]:
imputeAdata.obs['orig.ident'].unique()

In [ ]:
imputeAdata.var_names

In [ ]:
imputeAdata.X

In [ ]:
df = pd.DataFrame(imputeAdata.X)#.select_dtypes(include = ['number'])
df['orig.ident'] = imputeAdata.obs.reset_index()['orig.ident']

# Aggregation using Mean
df.groupby('orig.ident').mean()